# Exercise 2.4 — Missing data and imputation

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day2/day2-03a-imputation.html>.*

*Open it in Jupyter, VS Code or Positron and run it a cell at a time. Keep it in the folder that holds your `data` folder. Cells with `______` are blanks for you to fill in.*

## Overview

Lets understand what imputation is about, the difefrent ways in which we can apply it and the limitations it has. 

## The scenario

`hospital_admissions.csv` holds 250 emergency and elective admissions. The aim,
eventually, is to predict length of stay — but some of the information is
missing. Creatinine results are often not back yet when an emergency patient's
record is extracted, and some patients have no recorded smoking status.

[**Download `hospital_admissions.csv`**](data/hospital_admissions.csv) and save it in your `data` folder,
or take every file at once from the zip on the [introduction page](index.qmd). The same file is used in
Exercise 2.5 and Exercise 2.6.

### 1. Explore the missingness

Load the data and count what is missing.

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

hospital = pd.read_csv("data/hospital_admissions.csv")

print(hospital.dtypes)
print(hospital.isna().sum())

Creatinine is missing for 42 of the 250 patients and smoking status for 32.
Before deciding what to do about that, ask *who* is missing. Compare the
patients with and without a creatinine result:

In [ ]:
hospital["creatinine_missing"] = np.where(
    hospital["creatinine"].isna(), "Missing", "Observed")

table = pd.crosstab(hospital["admission_type"], hospital["creatinine_missing"])
print((100 * table / table.sum()).round(1))
print("p =", round(stats.chi2_contingency(table)[1], 3), "\n")

for col in ["age", "systolic_BP", "heart_rate"]:
    missing  = hospital.loc[hospital["creatinine_missing"] == "Missing", col]
    observed = hospital.loc[hospital["creatinine_missing"] == "Observed", col]
    print(f"{col:12s} {missing.mean():6.1f} ({missing.std():.1f})   "
          f"{observed.mean():6.1f} ({observed.std():.1f})   "
          f"p = {stats.ttest_ind(missing, observed).pvalue:.3f}")

> **Think about it**
>
> 1. Are the patients with missing creatinine systematically different from the
>    rest? In which variable?
> 2. What would happen to the cohort if you simply removed every patient with a
>    missing value?

------------------------------------------------------------------------

### 2. Simple imputation

#### A number: mean and median

The simplest thing to do with a missing number is to put the column's own
average in its place.

In [ ]:
hospital["creatinine_mean"] = hospital["creatinine"].fillna(
    hospital["creatinine"].mean())
hospital["creatinine_median"] = hospital["creatinine"].fillna(
    hospital["creatinine"].median())

print(round(hospital["creatinine"].mean(), 1))
print(hospital["creatinine"].median())

Now look at what that did to the distribution:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 2.8), sharey=True)
axes[0].hist(hospital["creatinine_mean"], bins=30, color="steelblue")
axes[0].set_title("mean imputed")
axes[1].hist(hospital["creatinine"].dropna(), bins=30, color="steelblue")
axes[1].set_title("observed")
for ax in axes:
    ax.set_xlabel("Creatinine (µmol/L)")
axes[0].set_ylabel("Patients")
fig.tight_layout()
plt.show()

print(round(hospital["creatinine"].std(), 1))
print(round(hospital["creatinine_mean"].std(), 1))

#### A category: the most common value, or an honest "Unknown"

Smoking status has three categories — Never, Former and Current — so there is
no mean to take. The usual substitute is the most frequent category. The
alternative is to say what you actually know.

In [ ]:
mode_smoking = hospital["smoking_status"].value_counts().index[0]

hospital["smoking_imputed"] = hospital["smoking_status"].fillna(mode_smoking)
hospital["smoking_unknown"] = hospital["smoking_status"].fillna("Unknown")

print(hospital["smoking_status"].value_counts(dropna=False).sort_index())
print(hospital["smoking_imputed"].value_counts().sort_index())
print(hospital["smoking_unknown"].value_counts().sort_index())

Before choosing between them, look at who the unknowns are:

In [ ]:
print(hospital.groupby("smoking_unknown").agg(
    patients=("patient_id", "size"),
    pct_emergency=("admission_type",
                   lambda s: round(100 * (s == "Emergency").mean()))))

> **Think about it**
>
> 1. What did mean imputation do to the creatinine histogram, and to its standard
>    deviation?
> 2. What happened to the smoking-status distribution after filling with the most
>    common category?
> 3. Would keeping "Unknown" as its own category be more appropriate here? Use
>    the last table to argue it.

------------------------------------------------------------------------

### 3. Using the other variables

Mean and median imputation give every patient the same number, whatever else is
known about them. But an 85-year-old emergency admission and a 30-year-old
elective one are unlikely to have the same creatinine. Can age, blood pressure
and heart rate do better?

`IterativeImputer` fits a regression of creatinine on the other three columns,
using the patients where it was measured, and predicts it where it was not. It
is still marked experimental in scikit-learn, so it has to be switched on.

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

features = ["age", "systolic_BP", "heart_rate", "creatinine"]

imputer = IterativeImputer(random_state=42)
hospital["creatinine_advanced"] = (
    imputer.fit_transform(hospital[features])[:, -1])

print(int(hospital["creatinine_advanced"].isna().sum()))

> **The same idea in R**
>
> The R tab does this job with the `mice` package. The two are the same idea —
> a regression on the other columns — so their answers land very close together
> but are not identical.

> **Think about it**
>
> Two patients of the same age, with the same blood pressure and heart rate, will
> be given the same imputed creatinine. Why might their real values be very
> different?

------------------------------------------------------------------------

### What to take from this

> **Imputation is making data up**
>
> Every value you filled in this exercise is an invention. A careful one, in the
> last case, but an invention — and once it is in the column it looks exactly like
> a measurement.

1. **Look at who is missing before deciding what to do.** Here the gaps tracked
   emergency admission, so dropping them would have quietly changed the cohort.
2. **Simple methods distort.** The mean put 42 patients on one value and
   shrank the spread; the mode turned 43% never-smokers into 56%.

### Optional: multiple imputation

The `mice` package is built for something more than the single fill used above.
Its default method does not give each gap one best guess; it draws a *plausible*
value, and it is meant to be run several times, so that the spread between the
filled-in datasets shows how uncertain each value was. That is multiple
imputation, and it is the honest way to carry imputation uncertainty through to
a final result. Read more at [amices.org/mice](https://amices.org/mice/).

Next: Exercise 2.5.